<p align="center">
  <img src="../assets/prodinno_logo.png" alt="Prodinno" width="200">
</p>

<h4 align="center">Session 3 · Hierarchical Clustering & Dendrogram</h4>
<h1 align="center">Hierarchical Clustering — Country Development Indicators</h1>
<p align="center"><i>Loading the country indicators dataset, understanding every column, and cleaning messy country-name duplicates</i></p>

---

## 1. The Business Problem: Prioritizing Aid Allocation

An international **NGO** has a fixed budget and needs to decide which countries need aid the
most urgently, and what *kind* of aid (health, economic, food security). Rather than judging
each of the 167 countries in isolation, the NGO wants to **group countries with similar
overall socio-economic development** into a handful of tiers, so that resources can be
targeted at a *cluster* level rather than country-by-country.

This is fundamentally different from the supervised problems you have seen so far
(regression, classification): there is **no label column** telling us "this country is
Tier 1" or "this country needs $X in aid." We only have raw socio-economic indicators, and
the task is to discover structure in the data itself — this is **unsupervised learning**.

> **Key distinction to keep in mind throughout this session:** in supervised learning, a
> wrong model is caught by comparing predictions to ground-truth labels. In unsupervised
> learning there is no ground truth to check against — the only safety net is *understanding
> the data itself* deeply enough to sanity-check whatever structure the algorithm finds.

We will build this out across four notebooks:

1. **`00_dataset_and_impurity`** *(this notebook)* — load the data, understand every column,
   and learn to detect and fix a very common real-world problem: duplicate entities recorded
   under slightly different spellings.
2. **`01_eda`** — exploratory data analysis of the (now clean) indicators.
3. **`02_data_processing`** — scaling and dimensionality reduction for distance-based
   clustering.
4. **`03_train_test_eval`** — hierarchical clustering, dendrograms, and cluster evaluation.

### 1.1 Loading the raw data

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

df_raw = pd.read_csv("data/country_data_raw.csv")
print("Shape:", df_raw.shape)
df_raw.head()

Shape: (167, 10)


,country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
0,Afghanistan,90.2,10.0,7.58,44.9,1610,9.44,56.2,5.82,553
1,Albania,16.6,28.0,6.55,48.6,9930,4.49,76.3,1.65,4090
2,Algeria,27.3,38.4,4.17,31.4,12900,16.10,76.5,2.89,4460
3,Angola,119.0,62.3,2.85,42.9,5900,22.40,60.1,6.16,3530
4,Antigua and Barbuda,10.3,45.5,6.03,58.9,19100,1.44,76.8,2.13,12200


In [2]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   country     167 non-null    str    
 1   child_mort  167 non-null    float64
 2   exports     167 non-null    float64
 3   health      167 non-null    float64
 4   imports     167 non-null    float64
 5   income      167 non-null    int64  
 6   inflation   167 non-null    float64
 7   life_expec  167 non-null    float64
 8   total_fer   167 non-null    float64
 9   gdpp        167 non-null    int64  
dtypes: float64(7), int64(2), str(1)
memory usage: 14.5 KB


In [3]:
df_raw.describe().T

,count,mean,std,min,25%,50%,75%,max
child_mort,167.0,38.270060,40.328931,2.6000,8.250,19.30,62.10,208.00
exports,167.0,41.108976,27.412010,0.1090,23.800,35.00,51.35,200.00
health,167.0,6.815689,2.746837,1.8100,4.920,6.32,8.60,17.90
imports,167.0,46.890215,24.209589,0.0659,30.200,43.30,58.75,174.00
income,167.0,17144.688623,19278.067698,609.0000,3355.000,9960.00,22800.00,125000.00
inflation,167.0,7.781832,10.570704,-4.2100,1.810,5.39,10.75,104.00
life_expec,167.0,70.555689,8.893172,32.1000,65.300,73.10,76.80,82.80
total_fer,167.0,2.947964,1.513848,1.1500,1.795,2.41,3.88,7.49
gdpp,167.0,12964.155689,18328.704809,231.0000,1330.000,4660.00,14050.00,105000.00


### 1.2 What each column means, and why the NGO cares

| Column | Meaning | Why it matters for aid allocation |
|---|---|---|
| `country` | Country name | The unit of analysis — one row per country |
| `child_mort` | Deaths of children under 5 per **1,000 live births** | Perhaps the single strongest proxy for the quality of a country's healthcare and nutrition system; directly signals urgent humanitarian need |
| `exports` | Exports of goods and services, **% of GDP** | Indicates how outward-facing / trade-dependent the economy is |
| `health` | Total health spending, **% of GDP** | Shows how much of a country's *own* resources go to healthcare — a low value alongside high `child_mort` is a red flag |
| `imports` | Imports of goods and services, **% of GDP** | Complements `exports`; together they describe trade openness |
| `income` | Net income per person, in **raw dollars per year** | A direct measure of average purchasing power — note this is on a totally different numeric scale (hundreds to tens of thousands) than most other columns |
| `inflation` | Annual growth rate of the **GDP deflator**, in % | High inflation erodes savings and purchasing power and often signals economic instability |
| `life_expec` | Average number of years a newborn would live under current mortality patterns | A summary statistic of overall public health and living conditions |
| `total_fer` | Average number of children per woman | Strongly (negatively) correlated with development level; also a planning input for future population needs |
| `gdpp` | **GDP per capita**, in raw dollars | The headline measure of economic output per person; historically the single most-cited "how rich is this country" number |

> **Blockquote — reading the scales carefully.** Notice already that `income` and `gdpp` are
> raw dollar amounts that can run into the tens of thousands, while `total_fer` and
> `health` are small numbers, often single digits. This difference in scale will become
> critical in `02_data_processing` when we build **distance-based** clusters — a naive
> Euclidean distance computed on unscaled data would be almost entirely driven by `income`
> and `gdpp`, silently ignoring everything else. Keep this observation in your back pocket.

The intuition the NGO uses: countries with **high `child_mort`**, **low `income`**, **low
`gdpp`**, and **low `life_expec`** are the ones most likely to need urgent humanitarian aid,
while countries with the opposite profile are more likely to be development partners or
donors rather than aid recipients. Clustering lets us discover *how many* meaningfully
different tiers exist in the data, rather than assuming a number up front.

### 1.3 Confirming the raw data is clean

Before we do anything else, let's verify the basic promise of this dataset: 167 rows, one
per country, no missing values.

In [4]:
print("Rows:", len(df_raw))
print("Unique countries:", df_raw["country"].nunique())
print("\nMissing values per column:")
print(df_raw.isna().sum())
print("\nDuplicate rows (exact):", df_raw.duplicated().sum())

Rows: 167
Unique countries: 167

Missing values per column:
country       0
child_mort    0
exports       0
health        0
imports       0
income        0
inflation     0
life_expec    0
total_fer     0
gdpp          0
dtype: int64



Duplicate rows (exact): 0


Confirmed: **167 rows, 167 unique country names, zero missing values, zero duplicate
rows.** This is our clean baseline. We will now deliberately corrupt a copy of this data to
practice a skill that is essential in any real NGO / government / multi-source dataset:
**detecting and fixing near-duplicate entity names.**

---

## 2. Injecting a Realistic Data Quality Problem

In practice, country-level data is stitched together from many sources (World Bank, UN,
regional development banks, national statistics offices), each with its own conventions for
spelling country names. It is extremely common to see the *same* country appear multiple
times under slightly different strings, for example:

- Extra **leading/trailing whitespace**: `"Afghanistan "`
- Inconsistent **capitalization**: `"ALBANIA"`
- Different **punctuation conventions**: `"Congo, Dem. Rep."` vs. `"Congo Dem Rep"`

If left undetected, this silently **inflates the number of "countries"** in the dataset and,
worse, can distort any aggregate statistic that treats each row as an independent country.
Let's simulate this by picking 7 real countries and appending a corrupted-name duplicate row
for each (same indicator values — only the `country` string is mangled).

In [5]:
rng = np.random.RandomState(42)

corruption_map = {
    "Afghanistan": "  Afghanistan  ",                  # leading/trailing whitespace
    "Zambia": "Zambia   ",                              # trailing whitespace
    "Albania": "ALBANIA",                                # ALL CAPS
    "Bahamas": "BAHAMAS",                                # ALL CAPS
    "Congo, Dem. Rep.": "Congo Dem Rep",                 # punctuation variant
    "Congo, Rep.": "Congo Rep",                          # punctuation variant
    "Micronesia, Fed. Sts.": "Micronesia Fed Sts",       # punctuation variant
}

corrupted_rows = []
for canonical_name, mangled_name in corruption_map.items():
    row = df_raw.loc[df_raw["country"] == canonical_name].copy()
    assert len(row) == 1, f"Expected exactly one match for {canonical_name}"
    row["country"] = mangled_name
    corrupted_rows.append(row)

df_dirty = pd.concat([df_raw] + corrupted_rows, ignore_index=True)
print("Dirty shape:", df_dirty.shape)
df_dirty.tail(10)

Dirty shape: (174, 10)


,country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
164,Vietnam,23.3,72.0,6.84,80.2,4490,12.100,73.1,1.95,1310
165,Yemen,56.3,30.0,5.18,34.4,4480,23.600,67.5,4.67,1310
166,Zambia,83.1,37.0,5.89,30.9,3280,14.000,52.0,5.40,1460
167,Afghanistan,90.2,10.0,7.58,44.9,1610,9.440,56.2,5.82,553
168,Zambia,83.1,37.0,5.89,30.9,3280,14.000,52.0,5.40,1460
169,ALBANIA,16.6,28.0,6.55,48.6,9930,4.490,76.3,1.65,4090
170,BAHAMAS,13.8,35.0,7.89,43.7,22900,-0.393,73.8,1.86,28000
171,Congo Dem Rep,116.0,41.1,7.91,49.6,609,20.800,57.5,6.54,334
172,Congo Rep,63.9,85.1,2.46,54.7,5190,20.700,60.4,4.95,2740
173,Micronesia Fed Sts,40.0,23.5,14.20,81.0,3340,3.800,65.4,3.46,2860


In [6]:
df_dirty.to_csv("data/country_data_dirty.csv", index=False)
print("Saved data/country_data_dirty.csv with", len(df_dirty), "rows")

Saved data/country_data_dirty.csv with 174 rows


---

## 3. Detecting the Problem

### 3.1 The `nunique()` count is now inflated

The simplest smell test: if we know the dataset should describe 167 countries, but
`nunique()` on the `country` column reports something higher, we have duplicate entities
hiding under different spellings.

In [7]:
print("Row count:", len(df_dirty))
print("Unique country strings (raw):", df_dirty["country"].nunique())
print("Extra 'countries' vs. the expected 167:", df_dirty["country"].nunique() - 167)

Row count: 174
Unique country strings (raw): 174
Extra 'countries' vs. the expected 167: 7


We have **7 more unique strings than real countries** — exactly matching the 7 rows we
injected. In a real project we would not know the "true" number in advance, so the next step
is to actually normalize the strings and see how much that count drops.

### 3.2 Normalizing whitespace and case

A large fraction of real-world name mismatches (whitespace, capitalization) can be fixed with
two simple string operations: `.str.strip()` to remove leading/trailing whitespace, and
`.str.lower()` to make casing consistent.

In [8]:
before = df_dirty["country"].nunique()

df_dirty["country_norm"] = df_dirty["country"].str.strip().str.lower()

after = df_dirty["country_norm"].nunique()

print(f"Unique country strings before normalization: {before}")
print(f"Unique country strings after strip().lower(): {after}")
print(f"Reduction: {before - after} duplicate strings collapsed")

Unique country strings before normalization: 174


Unique country strings after strip().lower(): 170
Reduction: 4 duplicate strings collapsed


Normalizing whitespace and case collapsed **4** of our 7 injected duplicates
(`Afghanistan`, `Zambia`, `Albania`, `Bahamas`) automatically, because those were purely
whitespace/casing corruptions. The remaining **3** (`Congo, Dem. Rep.` / `Congo, Rep.` /
`Micronesia, Fed. Sts.` and their punctuation-stripped variants) still count as separate
"countries" because normalizing case and whitespace does not touch punctuation — `"congo,
dem. rep."` and `"congo dem rep"` are still different strings.

### 3.3 Surfacing near-duplicate names with `difflib`

For the harder cases — where punctuation or wording differs — we can use
`difflib.get_close_matches` to flag pairs of strings that are *suspiciously similar*, so a
human can review them.

In [9]:
import difflib

unique_norm_names = sorted(df_dirty["country_norm"].unique())

candidates = []
for name in unique_norm_names:
    others = [n for n in unique_norm_names if n != name]
    matches = difflib.get_close_matches(name, others, n=3, cutoff=0.75)
    for match in matches:
        pair = tuple(sorted([name, match]))
        candidates.append(pair)

candidates = sorted(set(candidates))
print(f"{len(candidates)} near-duplicate name pairs flagged for review:\n")
for a, b in candidates:
    print(f"  {a!r:45s}  <->  {b!r}")

17 near-duplicate name pairs flagged for review:


  'argentina'                                    <->  'armenia'
  'australia'                                    <->  'austria'
  'brunei'                                       <->  'burundi'
  'congo dem rep'                                <->  'congo rep'
  'congo dem rep'                                <->  'congo, dem. rep.'
  'congo dem rep'                                <->  'congo, rep.'
  'congo rep'                                    <->  'congo, rep.'
  'congo, dem. rep.'                             <->  'congo, rep.'
  'gambia'                                       <->  'namibia'
  'gambia'                                       <->  'zambia'
  'iceland'                                      <->  'ireland'
  'iran'                                         <->  'iraq'
  'malawi'                                       <->  'mali'
  'micronesia fed sts'                           <->  'micronesia, fed. sts.'
  'namibia'                                      <->  'zambia'
  'niger'  

`difflib` correctly flags the three punctuation-based near-duplicates
(`"congo, dem. rep."` vs `"congo dem rep"`, etc.) for manual review. This is exactly the
workflow you would use on a real, messier dataset where you don't know in advance which
countries were corrupted.

> **Caution — near-duplicate names are a hint, not a verdict.** `get_close_matches` will also
> flag pairs of names that are similar **but refer to genuinely different countries** — for
> example `"Niger"` and `"Nigeria"`, or `"Slovakia"` and `"Slovenia"`, are textually close but
> are two entirely different nations with different indicator values. **Never merge two rows
> just because their names look alike.** The only safe rule is:
>
> $$\text{merge}(row_i, row_j) \iff \text{canonical\_name}(row_i) = \text{canonical\_name}(row_j) \;\; \text{AND} \;\; \text{values}(row_i) \approx \text{values}(row_j)$$
>
> i.e. **both** conditions must hold: the names must resolve to the same real-world entity
> *and* the numeric indicator values must be (near-)identical. Name similarity alone is never
> sufficient evidence — it is only a prompt to go check the values, and ideally an authoritative
> reference list of country names, before merging anything.

### 3.4 Checking whether "close name" pairs are actually the same row

Let's verify the caution above concretely: for every near-duplicate pair `difflib` found, are
the underlying indicator values actually identical (confirming they are duplicates), or could
they be different countries that merely sound alike?

In [10]:
cols_to_compare = ["child_mort", "exports", "health", "imports", "income",
                    "inflation", "life_expec", "total_fer", "gdpp"]

for a, b in candidates:
    row_a = df_dirty.loc[df_dirty["country_norm"] == a, cols_to_compare].iloc[0]
    row_b = df_dirty.loc[df_dirty["country_norm"] == b, cols_to_compare].iloc[0]
    identical = np.allclose(row_a.values, row_b.values)
    print(f"{a!r:35s} vs {b!r:35s} -> indicator values identical: {identical}")

'argentina'                         vs 'armenia'                           -> indicator values identical: False
'australia'                         vs 'austria'                           -> indicator values identical: False
'brunei'                            vs 'burundi'                           -> indicator values identical: False
'congo dem rep'                     vs 'congo rep'                         -> indicator values identical: False
'congo dem rep'                     vs 'congo, dem. rep.'                  -> indicator values identical: True
'congo dem rep'                     vs 'congo, rep.'                       -> indicator values identical: False
'congo rep'                         vs 'congo, rep.'                       -> indicator values identical: True
'congo, dem. rep.'                  vs 'congo, rep.'                       -> indicator values identical: False
'gambia'                            vs 'namibia'                           -> indicator values identical: 

'iceland'                           vs 'ireland'                           -> indicator values identical: False
'iran'                              vs 'iraq'                              -> indicator values identical: False
'malawi'                            vs 'mali'                              -> indicator values identical: False


'micronesia fed sts'                vs 'micronesia, fed. sts.'             -> indicator values identical: True
'namibia'                           vs 'zambia'                            -> indicator values identical: False
'niger'                             vs 'nigeria'                           -> indicator values identical: False
'pakistan'                          vs 'tajikistan'                        -> indicator values identical: False


All three flagged pairs have **identical indicator values**, confirming they really are
the same country recorded under a different spelling — safe to merge. Had any pair shown
different values, that would have been a strong signal they are two distinct countries and
must **not** be merged.

---

## 4. Fixing the Problem: Building a Canonical Name Mapping

Now we build an explicit mapping from every mangled string back to its canonical (correct)
country name. We deliberately do this **explicitly and by hand** rather than automatically
merging anything `difflib` flagged — this keeps a human in the loop for exactly the kind of
decision that should never be fully automated.

In [11]:
# Explicit canonical mapping: mangled string -> correct canonical country name.
# Built from the whitespace/case normalization (3.2) plus manual review of the
# difflib-flagged punctuation pairs (3.3), after confirming identical indicator values.
canonical_map = {
    "  afghanistan  ": "Afghanistan",
    "zambia": "Zambia",
    "albania": "Albania",
    "bahamas": "Bahamas",
    "congo dem rep": "Congo, Dem. Rep.",
    "congo rep": "Congo, Rep.",
    "micronesia fed sts": "Micronesia, Fed. Sts.",
}

def to_canonical(raw_name: str) -> str:
    norm = raw_name.strip().lower()
    if norm in canonical_map:
        return canonical_map[norm]
    # Fall back to a title-cased, whitespace-normalized version for names that
    # were never corrupted in the first place.
    return raw_name.strip()

df_dirty["country_canonical"] = df_dirty["country"].apply(to_canonical)

print("Unique canonical names:", df_dirty["country_canonical"].nunique())

Unique canonical names: 167


We are back to **167 unique canonical country names**. Now we drop the duplicate rows,
keeping one row per canonical name. Because we already confirmed (Section 3.4) that the
duplicated rows have identical indicator values, it does not matter which copy we keep.

In [12]:
df_clean = (
    df_dirty
    .drop_duplicates(subset="country_canonical", keep="first")
    .drop(columns=["country", "country_norm"])
    .rename(columns={"country_canonical": "country"})
    .sort_values("country")
    .reset_index(drop=True)
)

# Reorder columns to match the original schema
df_clean = df_clean[["country", "child_mort", "exports", "health", "imports", "income",
                      "inflation", "life_expec", "total_fer", "gdpp"]]

print("Cleaned shape:", df_clean.shape)
df_clean.head()

Cleaned shape: (167, 10)


,country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
0,Afghanistan,90.2,10.0,7.58,44.9,1610,9.44,56.2,5.82,553
1,Albania,16.6,28.0,6.55,48.6,9930,4.49,76.3,1.65,4090
2,Algeria,27.3,38.4,4.17,31.4,12900,16.10,76.5,2.89,4460
3,Angola,119.0,62.3,2.85,42.9,5900,22.40,60.1,6.16,3530
4,Antigua and Barbuda,10.3,45.5,6.03,58.9,19100,1.44,76.8,2.13,12200


### 4.1 Before/after verification

Two checks confirm the fix worked correctly:

1. **Unique country count restored to exactly 167.**
2. **Summary statistics of the numeric columns are unchanged** — since we only removed
   cosmetic duplicate rows (never touched a real value), `gdpp` and `income` mean/std must
   match the original raw dataset exactly.

In [13]:
print("Unique countries after cleaning:", df_clean["country"].nunique())
assert df_clean["country"].nunique() == 167
assert len(df_clean) == 167
print("Check passed: exactly 167 unique countries, 167 rows.")

Unique countries after cleaning: 167
Check passed: exactly 167 unique countries, 167 rows.


In [14]:
raw_sorted = df_raw.sort_values("country").reset_index(drop=True)
clean_sorted = df_clean.sort_values("country").reset_index(drop=True)

comparison = pd.DataFrame({
    "gdpp_mean_raw": [raw_sorted["gdpp"].mean()],
    "gdpp_mean_clean": [clean_sorted["gdpp"].mean()],
    "gdpp_std_raw": [raw_sorted["gdpp"].std()],
    "gdpp_std_clean": [clean_sorted["gdpp"].std()],
    "income_mean_raw": [raw_sorted["income"].mean()],
    "income_mean_clean": [clean_sorted["income"].mean()],
    "income_std_raw": [raw_sorted["income"].std()],
    "income_std_clean": [clean_sorted["income"].std()],
})
comparison

,gdpp_mean_raw,gdpp_mean_clean,gdpp_std_raw,gdpp_std_clean,income_mean_raw,income_mean_clean,income_std_raw,income_std_clean
0,12964.155689,12964.155689,18328.704809,18328.704809,17144.688623,17144.688623,19278.067698,19278.067698


In [15]:
assert np.isclose(raw_sorted["gdpp"].mean(), clean_sorted["gdpp"].mean())
assert np.isclose(raw_sorted["gdpp"].std(), clean_sorted["gdpp"].std())
assert np.isclose(raw_sorted["income"].mean(), clean_sorted["income"].mean())
assert np.isclose(raw_sorted["income"].std(), clean_sorted["income"].std())
assert df_raw.sort_values("country").reset_index(drop=True).equals(
    df_clean.sort_values("country").reset_index(drop=True)
)
print("Check passed: gdpp and income mean/std are unchanged, and the cleaned")
print("dataset is row-for-row identical to the original raw dataset.")

Check passed: gdpp and income mean/std are unchanged, and the cleaned
dataset is row-for-row identical to the original raw dataset.


In [16]:
df_clean.to_csv("data/country_data_clean.csv", index=False)
print("Saved data/country_data_clean.csv with", len(df_clean), "rows and",
      df_clean["country"].nunique(), "unique countries.")

Saved data/country_data_clean.csv with 167 rows and 167 unique countries.


---

## 5. Key Takeaways

- **Unsupervised learning has no labels to fall back on** — the only defense against data
  quality problems is understanding the data (what each column means, what the unit of
  analysis is, how many entities there *should* be) well enough to notice when something is
  off.
- A `nunique()` count higher than the number of real-world entities you expect is a strong
  signal of name-based duplication.
- `.str.strip().str.lower()` normalization catches whitespace and capitalization issues for
  free, but **not** punctuation or wording differences.
- `difflib.get_close_matches` (or similar fuzzy-matching tools) is useful for *surfacing*
  candidate duplicates, but **never** for automatically deciding they are duplicates — always
  verify the underlying values match before merging, and never merge on name similarity
  alone.
- After cleaning, always run a **before/after sanity check** on your summary statistics: if
  the fix only removed cosmetic duplicates, aggregate statistics like the mean and standard
  deviation of key numeric columns must be unchanged.

We now have a **clean 167-row dataset** (`data/country_data_clean.csv`) ready for
exploratory analysis in `01_eda.ipynb`.